# Multimodal Data Exploration

Our goal is to demonstrate how combining distinct data sources can reveal patterns. To do this, we will be looking at data from Michigan to see what we can discover.

We will be looking at two primary data sources: [**Public Elementary-Secondary Education Finance Data**](https://www2.census.gov/programs-surveys/school-finances/tables/2023/secondary-education-finance/elsec23.xlsx) and [**Michigan Cohort Graduation and Dropout Reports**](https://www.michigan.gov/cepi/-/media/Project/Websites/cepi/MISchoolData/2023-24/2024-Graduation-and-Dropout-Report-with-Subgroups.xlsx). To join these two data sources, we have to introduce a third data set from [**NCES CCD.**](https://nces.ed.gov/ccd/data/zip/ccd_lea_029_2223_w_1a_083023.zip)

## Loading Necessary Packages

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Loading Data

In [6]:
# Load in the graduation data and student info

grad = pd.read_excel(
    'data/michigan_school_info.xlsx',
    sheet_name='2024 4-Yr Grad Drop',
    skiprows=2
)

# Keep only district-level rows (no buildings, no statewide)
grad = grad[
    grad['BuildingCode'].isna() & 
    (grad['DistrictCode'] > 0)
].copy()

print(grad.shape)
grad.head()

(703, 18)


,District / Building Name (Code),Totals - \nFirst Time 9th Grade in Fall 2020,Totals - \n (+)Transfers In,Totals - \n (-)Transfers Out & Exempt,Totals - \nCohort Size,Cohort Status - \nNumber of On Time Graduates,Cohort Status - \nNumber of Dropouts,Cohort Status - \nNumber of Continuing in School,Cohort Status - \nNumber of Other Completers,Rates - \nGraduation Rate,Rates - \nDropout Rate,ISDCode,DistrictCode,BuildingCode,ISDName,DistrictName,BuildingName,Unnamed: 17
1,Adams Township School District (31020),49,7.0,4.0,52,46,< 10,< 10,< 10,0.8846,0.0192,31.0,31020.0,NaN,Copper Country ISD,Adams Township School District (31020),NaN,NaN
3,Addison Community Schools (46020),68,13.0,22.0,59,55,< 10,< 10,< 10,0.9322,0.0169,46.0,46020.0,NaN,Lenawee ISD,Addison Community Schools (46020),NaN,NaN
5,Adrian Public Schools (46010),221,41.0,44.0,218,178,28,12,< 10,0.8165,0.1284,46.0,46010.0,NaN,Lenawee ISD,Adrian Public Schools (46010),NaN,NaN
9,Airport Community Schools (58020),207,28.0,37.0,198,171,18,< 10,< 10,0.8636,0.0909,58.0,58020.0,NaN,Monroe ISD,Airport Community Schools (58020),NaN,NaN
14,Akron-Fairgrove Schools (79010),19,3.0,3.0,19,19,< 10,< 10,< 10,1.0000,0.0000,79.0,79010.0,NaN,Tuscola ISD,Akron-Fairgrove Schools (79010),NaN,NaN


In [7]:
# Load in the CCD data for the joins

ccd = pd.read_csv(
    'data/ccd_info.csv',
    encoding='latin1',
    low_memory=False
)

# Keep only Michigan and only the columns we need
ccd = ccd[ccd['FIPST'] == 26][['LEAID', 'ST_LEAID', 'LEA_NAME']].copy()

# Strip 'MI-' from ST_LEAID to get plain district code
ccd['dist_code'] = ccd['ST_LEAID'].str.replace('MI-', '', regex=False)

print(ccd.shape)
ccd.head()

(901, 4)


,LEAID,ST_LEAID,LEA_NAME,dist_code
7760,2600001,MI-84020,Michigan Department of Corrections,84020
7761,2600004,MI-84010,Michigan Department of Human Services,84010
7762,2600005,MI-13020,Battle Creek Public Schools,13020
7763,2600006,MI-27010,Bessemer Area School District,27010
7764,2600007,MI-32060,Harbor Beach Community Schools,32060


In [8]:
# Load in the district financial data

finance = pd.read_excel('data/census_financial.xlsx', sheet_name='elsec23')

# Keep only Michigan
finance = finance[finance['FIPST'] == 26].copy()

# Keep only the columns we need
finance = finance[[
    'NCESID', 'NAME', 'V33',       # ID and enrollment
    'TOTALREV', 'TFEDREV',          # total and federal revenue
    'TSTREV', 'TLOCREV',            # state and local revenue
    'TOTALEXP', 'TCURINST',         # total exp and instruction spending
    'TCURSSVC', 'TCAPOUT'           # support services and capital outlay
]].copy()

print(finance.shape)
finance.head()

(591, 11)


,NCESID,NAME,V33,TOTALREV,TFEDREV,TSTREV,TLOCREV,TOTALEXP,TCURINST,TCURSSVC,TCAPOUT
5274,2602160,ALCONA COMM SCHS,677,12739,1506,3237,7996,12406,6050,4613,1126
5275,2618570,HOPKINS PUBLIC SCHOOLS,1480,25220,1516,16673,7031,22700,12097,7720,1371
5276,2614230,FENNVILLE PUBLIC SCHOOLS,1247,23017,3185,12158,7674,22320,10920,7237,1525
5277,2635550,WAYLAND UNION SCHOOLS,2849,49763,3310,30354,16099,43035,23461,14105,657
5278,2680120,ALPENA-MONTMORENCY-ALCONA SCHOOL DIST,0,16997,2477,7762,6758,14689,1612,9585,290


A couple of things to note before we merge:

`grad` has 703 districts

`ccd` has 901 Michigan entries — more than `grad` because it includes all LEA types (intermediate districts, state agencies, etc.), not just regular districts

`finance` has 591 — fewer because not every Michigan district reports to the Census finance survey

In [5]:
# Prep the district code in grad to match ccd
grad['dist_code'] = grad['DistrictCode'].astype(int).astype(str).str.zfill(5)

# Merge grad with ccd to get the LEAID
grad_ccd = grad.merge(
    ccd[['LEAID', 'dist_code']],
    on='dist_code',
    how='inner'
)

print(grad_ccd.shape)
grad_ccd.head()

(703, 20)


,District / Building Name (Code),Totals - \nFirst Time 9th Grade in Fall 2020,Totals - \n (+)Transfers In,Totals - \n (-)Transfers Out & Exempt,Totals - \nCohort Size,Cohort Status - \nNumber of On Time Graduates,Cohort Status - \nNumber of Dropouts,Cohort Status - \nNumber of Continuing in School,Cohort Status - \nNumber of Other Completers,Rates - \nGraduation Rate,Rates - \nDropout Rate,ISDCode,DistrictCode,BuildingCode,ISDName,DistrictName,BuildingName,Unnamed: 17,dist_code,LEAID
0,Adams Township School District (31020),49,7.0,4.0,52,46,< 10,< 10,< 10,0.8846,0.0192,31.0,31020.0,NaN,Copper Country ISD,Adams Township School District (31020),NaN,NaN,31020,2601890
1,Addison Community Schools (46020),68,13.0,22.0,59,55,< 10,< 10,< 10,0.9322,0.0169,46.0,46020.0,NaN,Lenawee ISD,Addison Community Schools (46020),NaN,NaN,46020,2601920
2,Adrian Public Schools (46010),221,41.0,44.0,218,178,28,12,< 10,0.8165,0.1284,46.0,46010.0,NaN,Lenawee ISD,Adrian Public Schools (46010),NaN,NaN,46010,2601950
3,Airport Community Schools (58020),207,28.0,37.0,198,171,18,< 10,< 10,0.8636,0.0909,58.0,58020.0,NaN,Monroe ISD,Airport Community Schools (58020),NaN,NaN,58020,2601980
4,Akron-Fairgrove Schools (79010),19,3.0,3.0,19,19,< 10,< 10,< 10,1.0000,0.0000,79.0,79010.0,NaN,Tuscola ISD,Akron-Fairgrove Schools (79010),NaN,NaN,79010,2602010


In [9]:
# LEAID and NCESID are the same thing, just need matching types
grad_ccd['LEAID'] = grad_ccd['LEAID'].astype(str)
finance['NCESID'] = finance['NCESID'].astype(str)

# Final merge
michigan_merged = grad_ccd.merge(
    finance,
    left_on='LEAID',
    right_on='NCESID',
    how='inner'
)

print(michigan_merged.shape)
michigan_merged.head()

(563, 31)


,District / Building Name (Code),Totals - \nFirst Time 9th Grade in Fall 2020,Totals - \n (+)Transfers In,Totals - \n (-)Transfers Out & Exempt,Totals - \nCohort Size,Cohort Status - \nNumber of On Time Graduates,Cohort Status - \nNumber of Dropouts,Cohort Status - \nNumber of Continuing in School,Cohort Status - \nNumber of Other Completers,Rates - \nGraduation Rate,...,NAME,V33,TOTALREV,TFEDREV,TSTREV,TLOCREV,TOTALEXP,TCURINST,TCURSSVC,TCAPOUT
0,Adams Township School District (31020),49,7.0,4.0,52,46,< 10,< 10,< 10,0.8846,...,ADAMS TWP SCH DIST,466,7156,746,4940,1470,6852,4259,1743,175
1,Addison Community Schools (46020),68,13.0,22.0,59,55,< 10,< 10,< 10,0.9322,...,ADDISON COMM SCH DIST,717,12187,1511,4910,5766,11317,6502,3459,285
2,Adrian Public Schools (46010),221,41.0,44.0,218,178,28,12,< 10,0.8165,...,ADRIAN CITY SCH DIST,2771,53902,11407,30039,12456,52329,26215,15260,5710
3,Airport Community Schools (58020),207,28.0,37.0,198,171,18,< 10,< 10,0.8636,...,AIRPORT COMM SCH DIST,2809,46655,4981,30867,10807,43264,23392,11896,4638
4,Akron-Fairgrove Schools (79010),19,3.0,3.0,19,19,< 10,< 10,< 10,1.0000,...,AKRON-FAIRGROVE SCH DIST,340,7574,1840,3032,2702,7057,3242,2346,441
